Part A will not be done for now.I have planned to do improvements in a couple of days.Thank you for understandings!

### B1

In [2]:
from pathlib import Path

# Create all folders
Path("container/app").mkdir(parents=True, exist_ok=True)
Path("container/models").mkdir(parents=True, exist_ok=True)

print("Folders created:")
for p in sorted(Path("container").rglob("*")):
    print(f"  {p}")

Folders created:
  container\.ipynb_checkpoints
  container\.ipynb_checkpoints\STUDENT-checkpoint.json
  container\app
  container\app\__init__.py
  container\app\cli.py
  container\Dockerfile
  container\models
  container\models\best.onnx
  container\requirements.txt
  container\STUDENT.json


### B2

In [3]:
import json

student = {
    "first_name": "Mahammad",
    "last_name": "Sadigov",
    "team": "mahammad-sadigov",
    "model": {
        "framework": "yolo26",
        "variant": "yolo26n",
        "imgsz": 640,
        "epochs_total": 30,
        "tricks": []
    },
    "notes": "In my opinion,It is not completed.I will try to improve the model"
}

Path("container/STUDENT.json").write_text(json.dumps(student, indent=2))
print("STUDENT.json written:")
print(json.dumps(student, indent=2))

STUDENT.json written:
{
  "first_name": "Mahammad",
  "last_name": "Sadigov",
  "team": "mahammad-sadigov",
  "model": {
    "framework": "yolo26",
    "variant": "yolo26n",
    "imgsz": 640,
    "epochs_total": 30,
    "tricks": []
  },
  "notes": "In my opinion,It is not completed.I will try to improve the model"
}


### B3

In [4]:
cli_code = '''import sys
import json
import csv
from pathlib import Path
from detector import CatDetector

STUDENT_PATH = Path("/app/STUDENT.json")
MODEL_PATH   = Path("/app/models/best.onnx")
INPUT_DIR    = Path("/data/input")
OUTPUT_DIR   = Path("/data/output")
IMG_EXTS     = {".jpg", ".jpeg", ".png"}


def cmd_info():
    print(STUDENT_PATH.read_text())


def cmd_predict():
    detector = CatDetector(str(MODEL_PATH))

    image_paths = sorted([
        p for p in INPUT_DIR.rglob("*")
        if p.suffix.lower() in IMG_EXTS
    ])

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    out_csv = OUTPUT_DIR / "predictions.csv"

    with open(out_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["image_path", "xmin", "ymin", "xmax", "ymax", "confidence", "class"])

        for img_path in image_paths:
            rel_path = img_path.relative_to(INPUT_DIR).as_posix()
            detections = detector.predict(str(img_path))

            if not detections:
                writer.writerow([rel_path, "", "", "", "", "", ""])
            else:
                for d in detections:
                    writer.writerow([
                        rel_path,
                        round(d["xmin"], 2),
                        round(d["ymin"], 2),
                        round(d["xmax"], 2),
                        round(d["ymax"], 2),
                        round(d["confidence"], 4),
                        d["class"],
                    ])

    print(f"predictions.csv written -> {out_csv}  ({len(image_paths)} images processed)")


if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("Usage: python cli.py [info|predict]")
        sys.exit(1)

    cmd = sys.argv[1].lower()
    if cmd == "info":
        cmd_info()
    elif cmd == "predict":
        cmd_predict()
    else:
        print(f"Unknown command: {cmd}")
        sys.exit(1)
'''

Path("container/app/cli.py").write_text(cli_code, encoding="utf-8")
print("cli.py rewritten successfully.")

cli.py rewritten successfully.


### B4

In [5]:
# Write __init__.py (empty file, makes app/ a Python package)
Path("container/app/__init__.py").write_text("")
print("__init__.py written.")

# Write Dockerfile
dockerfile = '''FROM python:3.11-slim

WORKDIR /app

COPY container/requirements.txt /app/requirements.txt
RUN pip install --no-cache-dir -r /app/requirements.txt

COPY container/app /app/app
COPY container/models /app/models
COPY container/STUDENT.json /app/STUDENT.json

ENTRYPOINT ["python", "/app/app/cli.py"]
'''

Path("container/Dockerfile").write_text(dockerfile)
print("Dockerfile written.")

__init__.py written.
Dockerfile written.


### B5

## Export best.pt to ONNX

We export the best checkpoint from m6-04 training to ONNX format.
YOLO26's end-to-end NMS-free design means the export is clean —
no separate post-processing needed at inference time.

In [6]:
%pip install ultralytics

Note: you may need to restart the kernel to use updated packages.


In [7]:
from ultralytics import YOLO
from pathlib import Path

# Point this at your best checkpoint from m6-04
best_pt = Path(r"C:\Users\user\BootcampLabs\m6-04-assessment\m6-04-assessment\runs\detect\runs\cats_v1\weights\best.pt")

model = YOLO(str(best_pt))

# Export to ONNX
model.export(
    format="onnx",
    imgsz=640,
    opset=17,
    dynamic=False,
)

print("Export done!")

Ultralytics 8.4.58  Python-3.11.15 torch-2.12.0+cpu CPU (12th Gen Intel Core i5-1235U)
YOLO26n summary (fused): 122 layers, 2,375,031 parameters, 0 gradients, 5.2 GFLOPs

PyTorch: starting from 'C:\Users\user\BootcampLabs\m6-04-assessment\m6-04-assessment\runs\detect\runs\cats_v1\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (5.1 MB)

ONNX: starting export with onnx 1.21.0 opset 17...


C:\Users\user\anaconda3\envs\pytorch_env\Lib\site-packages\torch\onnx\_internal\torchscript_exporter\symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 17 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)


ONNX: slimming with onnxslim 0.1.94...
ONNX: export success  2.1s, saved as 'C:\Users\user\BootcampLabs\m6-04-assessment\m6-04-assessment\runs\detect\runs\cats_v1\weights\best.onnx' (9.4 MB)

Export complete (2.6s)
Results saved to C:\Users\user\BootcampLabs\m6-04-assessment\m6-04-assessment\runs\detect\runs\cats_v1\weights\best.onnx
Predict:         yolo predict task=detect model=C:\Users\user\BootcampLabs\m6-04-assessment\m6-04-assessment\runs\detect\runs\cats_v1\weights\best.onnx imgsz=640 
Validate:        yolo val task=detect model=C:\Users\user\BootcampLabs\m6-04-assessment\m6-04-assessment\runs\detect\runs\cats_v1\weights\best.onnx imgsz=640 data=C:\Users\user\BootcampLabs\m6-04-assessment\m6-04-assessment\data.yaml  
Visualize:       https://netron.app
Export done!


In [8]:
import shutil

onnx_src = Path(r"C:\Users\user\BootcampLabs\m6-04-assessment\m6-04-assessment\runs\detect\runs\cats_v1\weights\best.onnx")
onnx_dst = Path("container/models/best.onnx")

shutil.copy(onnx_src, onnx_dst)
print(f"Copied {onnx_src} -> {onnx_dst}")
print(f"File size: {onnx_dst.stat().st_size / 1e6:.1f} MB")

Copied C:\Users\user\BootcampLabs\m6-04-assessment\m6-04-assessment\runs\detect\runs\cats_v1\weights\best.onnx -> container\models\best.onnx
File size: 9.8 MB


We load best.onnx with onnxruntime and run inference on a few test images.
We then compare the boxes against the original PyTorch model to confirm
the export is correct within a small numerical tolerance.

In [9]:
import onnxruntime as ort
import numpy as np
from PIL import Image
from ultralytics import YOLO
from pathlib import Path

# ── Load ONNX session ──────────────────────────────────────
onnx_path = Path("container/models/best.onnx")
session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
input_name = session.get_inputs()[0].name

print(f"Input  : {session.get_inputs()[0].name}  {session.get_inputs()[0].shape}")
print(f"Output : {session.get_outputs()[0].name}  {session.get_outputs()[0].shape}")

# ── Load PyTorch model ─────────────────────────────────────
best_pt = Path(r"C:\Users\user\BootcampLabs\m6-04-assessment\m6-04-assessment\runs\detect\runs\cats_v1\weights\best.pt")
pt_model = YOLO(str(best_pt))

# ── Pick 3 test images ─────────────────────────────────────
BASE = Path(r"C:\Users\user\BootcampLabs\m6-04-assessment\m6-04-assessment\DATA_CLEAN-20260521T192120Z-3-001\DATA_CLEAN\_splits")
test_imgs = sorted((BASE / "test" / "images").glob("*.jpg"))[:3]
print(f"\nTesting on {len(test_imgs)} images")

Input  : images  [1, 3, 640, 640]
Output : output0  [1, 300, 6]

Testing on 3 images


In [10]:
def letterbox(img, size=640):
    orig_w, orig_h = img.size
    scale = min(size / orig_w, size / orig_h)
    new_w, new_h = int(orig_w * scale), int(orig_h * scale)
    img_r = img.resize((new_w, new_h), Image.BILINEAR)
    pad_x = (size - new_w) // 2
    pad_y = (size - new_h) // 2
    padded = Image.new("RGB", (size, size), (114, 114, 114))
    padded.paste(img_r, (pad_x, pad_y))
    return padded, scale, (pad_x, pad_y)

for img_path in test_imgs:
    img = Image.open(img_path).convert("RGB")
    x, scale, (pad_x, pad_y) = letterbox(img)
    x_np = (np.array(x, dtype=np.float32) / 255.0).transpose(2, 0, 1)[None, ...]

    # ONNX inference
    onnx_out = session.run(None, {input_name: x_np})[0][0]  # (300, 6)
    onnx_dets = onnx_out[onnx_out[:, 4] > 0.25]

    # PyTorch inference
    pt_out = pt_model.predict(str(img_path), imgsz=640, conf=0.25, verbose=False)
    pt_dets = pt_out[0].boxes.xyxy.numpy() if pt_out[0].boxes else []

    print(f"\n{img_path.name}")
    print(f"  ONNX detections : {len(onnx_dets)}")
    print(f"  PyTorch dets    : {len(pt_dets)}")
    if len(onnx_dets) > 0 and len(pt_dets) > 0:
        print(f"  ONNX   box[0]   : {onnx_dets[0][:4].round(1)}")
        print(f"  PyTorch box[0]  : {pt_dets[0].round(1)}")


008e31f109e2c872.jpg
  ONNX detections : 1
  PyTorch dets    : 1
  ONNX   box[0]   : [       15.1         105       444.9         518]
  PyTorch box[0]  : [       71.8           0      1346.2      1235.7]

01afee909d184eae.jpg
  ONNX detections : 1
  PyTorch dets    : 1
  ONNX   box[0]   : [        158        89.7       450.5       582.5]
  PyTorch box[0]  : [      265.7       148.3       759.9       986.8]

01c63262502ae73b.jpg
  ONNX detections : 2
  PyTorch dets    : 2
  ONNX   box[0]   : [       11.9       133.9       346.6       618.6]
  PyTorch box[0]  : [       43.4       598.1      1536.4      2743.3]


----

I am trying to solve the problems in docker below

detector.py was not created

In [11]:
dockerfile = '''FROM python:3.11-slim

WORKDIR /app

COPY container/requirements.txt /app/requirements.txt
RUN pip install --no-cache-dir -r /app/requirements.txt

COPY container/app /app/app
COPY container/models /app/models
COPY container/STUDENT.json /app/STUDENT.json

WORKDIR /app/app

ENTRYPOINT ["python", "/app/app/cli.py"]
'''

Path("container/Dockerfile").write_text(dockerfile, encoding="utf-8")
print("Dockerfile rewritten successfully.")

Dockerfile rewritten successfully.


In [12]:
cli_code = '''import sys
import json
import csv
from pathlib import Path

sys.path.insert(0, "/app/app")
from detector import CatDetector

STUDENT_PATH = Path("/app/STUDENT.json")
MODEL_PATH   = Path("/app/models/best.onnx")
INPUT_DIR    = Path("/data/input")
OUTPUT_DIR   = Path("/data/output")
IMG_EXTS     = {".jpg", ".jpeg", ".png"}


def cmd_info():
    print(STUDENT_PATH.read_text())


def cmd_predict():
    detector = CatDetector(str(MODEL_PATH))

    image_paths = sorted([
        p for p in INPUT_DIR.rglob("*")
        if p.suffix.lower() in IMG_EXTS
    ])

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    out_csv = OUTPUT_DIR / "predictions.csv"

    with open(out_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["image_path", "xmin", "ymin", "xmax", "ymax", "confidence", "class"])

        for img_path in image_paths:
            rel_path = img_path.relative_to(INPUT_DIR).as_posix()
            detections = detector.predict(str(img_path))

            if not detections:
                writer.writerow([rel_path, "", "", "", "", "", ""])
            else:
                for d in detections:
                    writer.writerow([
                        rel_path,
                        round(d["xmin"], 2),
                        round(d["ymin"], 2),
                        round(d["xmax"], 2),
                        round(d["ymax"], 2),
                        round(d["confidence"], 4),
                        d["class"],
                    ])

    print(f"predictions.csv written -> {out_csv}  ({len(image_paths)} images processed)")


if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("Usage: python cli.py [info|predict]")
        sys.exit(1)

    cmd = sys.argv[1].lower()
    if cmd == "info":
        cmd_info()
    elif cmd == "predict":
        cmd_predict()
    else:
        print(f"Unknown command: {cmd}")
        sys.exit(1)
'''

Path("container/app/cli.py").write_text(cli_code, encoding="utf-8")
print("cli.py rewritten successfully.")

cli.py rewritten successfully.


In [13]:
import os
print(os.listdir("container/app"))

['cli.py', '__init__.py']


In [14]:
detector_code = '''import numpy as np
import onnxruntime as ort
from PIL import Image


class CatDetector:
    def __init__(self, onnx_path, imgsz=640, conf=0.25, class_names=("cat",)):
        self.session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
        self.imgsz = imgsz
        self.conf = conf
        self.class_names = class_names
        self.input_name = self.session.get_inputs()[0].name

    def _letterbox(self, img, size):
        orig_w, orig_h = img.size
        scale = min(size / orig_w, size / orig_h)
        new_w = int(orig_w * scale)
        new_h = int(orig_h * scale)
        img = img.resize((new_w, new_h), Image.BILINEAR)
        pad_x = (size - new_w) // 2
        pad_y = (size - new_h) // 2
        padded = Image.new("RGB", (size, size), (114, 114, 114))
        padded.paste(img, (pad_x, pad_y))
        return padded, scale, (pad_x, pad_y)

    def predict(self, image_path: str) -> list:
        img = Image.open(image_path).convert("RGB")
        orig_w, orig_h = img.size

        x, scale, (pad_x, pad_y) = self._letterbox(img, self.imgsz)
        x = (np.array(x, dtype=np.float32) / 255.0).transpose(2, 0, 1)[None, ...]

        out = self.session.run(None, {self.input_name: x})[0]
        out = out[0]

        results = []
        for x1, y1, x2, y2, score, cls in out:
            if score < self.conf:
                continue
            x1 = (x1 - pad_x) / scale
            y1 = (y1 - pad_y) / scale
            x2 = (x2 - pad_x) / scale
            y2 = (y2 - pad_y) / scale
            x1 = max(0.0, min(orig_w, float(x1)))
            y1 = max(0.0, min(orig_h, float(y1)))
            x2 = max(0.0, min(orig_w, float(x2)))
            y2 = max(0.0, min(orig_h, float(y2)))
            results.append({
                "xmin": x1, "ymin": y1,
                "xmax": x2, "ymax": y2,
                "confidence": float(score),
                "class": self.class_names[int(cls)],
            })
        return results
'''

Path("container/app/detector.py").write_text(detector_code, encoding="utf-8")
print(os.listdir("container/app"))

['cli.py', 'detector.py', '__init__.py']


now detector.pyis loading,but onnxruntime is incompatible with numpy

In [15]:
requirements = """onnxruntime==1.18.0
numpy<2.0
pillow
opencv-python-headless
"""

Path("container/requirements.txt").write_text(requirements, encoding="utf-8")
print("requirements.txt updated.")

requirements.txt updated.


In [16]:
readme = """# m6-09-assessment — Cat Detection v2

## Image for leaderboard
docker pull maqa333/cat-detector:final
Image: maqa333/cat-detector:final
Student: Mahammad Sadigov

## Run

### info
docker run --rm maqa333/cat-detector:final info

### predict
docker run --rm \\
  -v /absolute/path/to/images:/data/input:ro \\
  -v /absolute/path/to/results:/data/output \\
  maqa333/cat-detector:final predict

Output is written to /data/output/predictions.csv

## Dataset
Same cat detection dataset from m6-04-assessment.
Split: 70/15/15 train/val/test.

## Model
- Framework: YOLO26
- Variant: yolo26n
- Epochs: 30
- mAP@0.5: 0.8353
- mAP@0.5:0.95: 0.5394
"""

Path("README.md").write_text(readme, encoding="utf-8")
print("README.md written.")

README.md written.
